# Demo: Representing Time-Series Data in pandas

We've got a CSV with three years of daily bike-share rides. Right now it's just rows and columns — pandas doesn't know there's anything special about the dates. Let's fix that, and then see what we can do once pandas actually understands the time dimension.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Loading the data — before and after

Let's load the CSV the "naive" way first, so you can see what it looks like when pandas doesn't know the dates are dates.

In [ ]:
# The naive way — just read the CSV
df_raw = pd.read_csv("../data/bikeshare_rides.csv")
print(df_raw.dtypes)
print()
df_raw.head()

See that? The `date` column is type `object` — that's just a string. pandas has no idea these are dates. It can't sort them chronologically, it can't resample by week, nothing. It's just text.

Now let's load it properly with `parse_dates`.

In [ ]:
df = pd.read_csv("../data/bikeshare_rides.csv", parse_dates=["date"])
print(df.dtypes)
print()
df.head()

Now `date` is `datetime64`. That's the magic type. But we're not done — it's still just a regular column. We want it as the **index**.

## Setting a datetime index

This is the single most important step for time-series work in pandas. Once the dates are the index, everything unlocks — `.resample()`, `.asfreq()`, slicing by year like `df["2023"]`. Without it, none of that works.

In [ ]:
df = df.set_index("date")
print(f"Index type: {type(df.index).__name__}")
print(f"Range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Total rows: {len(df)}")
df.head()

Notice the index changed — it's no longer 0, 1, 2... it's actual dates. And we can do cool stuff now, like grab all of 2023 with a single slice:

In [ ]:
# Slice by year — only works with a DatetimeIndex
print(f"2023 only: {len(df['2023'])} days")
print(f"Summer 2022: {len(df['2022-06':'2022-08'])} days")

## Resampling — changing the frequency

We have daily data. But what if we need weekly totals? Monthly? That's what `.resample()` does — it groups by time buckets and applies an aggregation. Think of it like `groupby` but for dates.

In [ ]:
weekly = df["rides"].resample("W").sum()
monthly = df["rides"].resample("MS").sum()

print(f"Daily:   {len(df)} rows")
print(f"Weekly:  {len(weekly)} rows")
print(f"Monthly: {len(monthly)} rows")

Let's plot all three so you can see the difference. Daily is noisy — lots of day-to-day spikes. Weekly smooths that out. Monthly gives you just the big seasonal shape.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(df.index, df["rides"], linewidth=0.5)
axes[0].set_ylabel("Rides")
axes[0].set_title("Daily")

axes[1].plot(weekly.index, weekly.values, linewidth=0.8, color="tab:orange")
axes[1].set_ylabel("Rides")
axes[1].set_title("Weekly")

axes[2].plot(monthly.index, monthly.values, linewidth=1.2, color="tab:green")
axes[2].set_ylabel("Rides")
axes[2].set_title("Monthly")

plt.tight_layout()
plt.show()

Same data, three different views. The seasonal pattern — more rides in summer, fewer in winter — is obvious in the monthly plot but hard to see in the daily one. Which frequency you use depends on what question you're answering.

## Checking for gaps in the data

Here's something that bites people: your data might have missing dates. Maybe the system was down for a day, maybe there was a holiday with no recording. Most time-series models assume a regular frequency — if there are gaps, things break.

The trick is to build the *complete* date range from first to last date, then compare it to what we actually have.

In [ ]:
full_range = pd.date_range(df.index.min(), df.index.max(), freq="D")
missing = full_range.difference(df.index)

print(f"Expected days (complete range): {len(full_range)}")
print(f"Actual days in data:            {len(df)}")
print(f"Missing days:                   {len(missing)}")

if len(missing) > 0:
    print(f"\nFirst few missing dates:")
    for d in missing[:5]:
        print(f"  {d.date()}")

Good to know. Before modeling, you'd want to fill those gaps — either interpolate or forward-fill. But at least now you know they exist. That's the whole point of this step: don't assume your data is clean.